# Zip All .srt Files

This notebook finds all `.srt` (subtitle) files in a specified folder and creates a zip archive containing them.


In [ ]:
import os
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime

# Mount Google Drive (if not already mounted)
try:
    from google.colab import drive
    drive_path = '/content/drive'
    if not os.path.exists(drive_path):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        print("✓ Google Drive mounted successfully")
    else:
        print("✓ Google Drive is already mounted")
except ImportError:
    print("Note: Not running in Google Colab, skipping Drive mount")
except Exception as e:
    print(f"Warning: Could not mount Google Drive: {e}")

# Define paths for the directories
base_folder = '/content/drive/My Drive/Media for transcription'
to_process_folder = os.path.join(base_folder, 'To be processed')
processed_folder = os.path.join(base_folder, 'Processed')
processed_and_ingested_folder = os.path.join(base_folder, 'Processed and ingested')

# Configuration
FOLDER_PATH = processed_folder
OUTPUT_ZIP_NAME = f"srt_files_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"  # Auto-generated name with timestamp

print(f"\nSearching for .srt files in: {FOLDER_PATH}")
print(f"Output zip file: {OUTPUT_ZIP_NAME}")


In [ ]:
# Check if folder exists and find all .srt files
folder = Path(FOLDER_PATH)

if not folder.exists():
    print(f"❌ Error: Folder does not exist: {FOLDER_PATH}")
    print(f"   Please check:")
    print(f"   1. Is Google Drive mounted?")
    print(f"   2. Does the path exist?")
    print(f"   3. Check the folder name spelling")
    srt_files = []
else:
    print(f"✓ Folder exists: {FOLDER_PATH}")
    srt_files = list(folder.rglob("*.srt"))
    
    if not srt_files:
        print(f"No .srt files found in the specified folder.")
        print(f"   Searched recursively in: {FOLDER_PATH}")
    else:
        print(f"Found {len(srt_files)} .srt file(s):")
        for srt_file in srt_files:
            print(f"  - {srt_file}")


In [ ]:
# Simple: just zip the files
if srt_files:
    from zipfile import ZIP_STORED
    from collections import Counter
    
    print(f"\nZipping {len(srt_files)} files...")
    
    duplicate_names = Counter()
    
    with ZipFile(OUTPUT_ZIP_NAME, 'w', compression=ZIP_STORED) as zipf:
        for idx, srt_file in enumerate(srt_files, 1):
            arcname = srt_file.name
            
            # Handle duplicates
            if arcname in duplicate_names:
                duplicate_names[arcname] += 1
                name_parts = arcname.rsplit('.', 1)
                arcname = f"{name_parts[0]}_{duplicate_names[arcname]}.{name_parts[1]}"
            else:
                duplicate_names[arcname] = 1
            
            zipf.write(srt_file, arcname)
            
            if idx % 100 == 0:
                print(f"  {idx}/{len(srt_files)} files...")
    
    zip_size = os.path.getsize(OUTPUT_ZIP_NAME) / (1024 * 1024)
    print(f"\n✓ Done! Zip file: {OUTPUT_ZIP_NAME} ({zip_size:.2f} MB)")
    print(f"  Download it from the Colab file browser on the left")
else:
    print("No files to zip.")


In [ ]:
# Download the zip file
import os
import glob

# Try to find the zip file
zip_file = None

# First, try the variable (if it exists)
try:
    if 'OUTPUT_ZIP_NAME' in globals() and os.path.exists(OUTPUT_ZIP_NAME):
        zip_file = OUTPUT_ZIP_NAME
except:
    pass

# If not found, look for the most recent srt_files zip
if not zip_file:
    zip_files = glob.glob('/content/srt_files_*.zip')
    if zip_files:
        zip_file = max(zip_files, key=os.path.getmtime)
        print(f"Found zip file: {zip_file}")

if zip_file and os.path.exists(zip_file):
    try:
        from google.colab import files
        print(f"Downloading {os.path.basename(zip_file)}...")
        files.download(zip_file)
        print("✓ Download started!")
    except ImportError:
        print(f"File location: {os.path.abspath(zip_file)}")
        print("Not in Colab - file is saved locally")
else:
    print(f"❌ Zip file not found")
    print("Location should be: /content/srt_files_*.zip")
    print("Or check the Colab file browser on the left sidebar")
